# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [01:16<00:00, 15.25s/it]


In [4]:
len(deals)

50

In [5]:
deals[0].describe()

"Title: Refurbished Unlocked Apple iPhone 12 Pro Max 256GB Smartphone for $400 + free shipping\nDetails: That's $52 under our November mention of a 128GB (this one is 256GB) and the lowest price we've seen in any condition. A 1-year Allstate warranty applies. Buy Now at eBay\nFeatures: \nURL: https://www.dealnews.com/products/Apple/Unlocked-Apple-iPhone-12-Pro-Max-128-GB-Smartphone/314138.html?iref=rss-c142"

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Refurbished Unlocked Apple iPhone 12 Pro Max 256GB Smartphone for $400 + free shipping
Details: That's $52 under our November mention of a 128GB (this one is 256GB) and the lowest price we've seen in any condition. A 1-year Allstate warranty applies. Buy Now at eBay
Features: 
URL: https://www.dealnews.com/products/Apple/Unlocked-Apple-iPhone-12-Pro-Max-128-GB-Smart

In [13]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [14]:
result = get_recommendations()

In [15]:
len(result.deals)

5

In [16]:
result.deals[1]

Deal(product_description='The Refurbished Unlocked Samsung Galaxy S22 Ultra is a powerhouse featuring a massive 512GB of internal storage. Its 6.8-inch Dynamic AMOLED display boasts a 120Hz refresh rate for ultra-smooth scrolling and vibrant visuals. This smartphone houses a versatile camera system, allowing for stunning photography and videography, and is powered by a high-performance Snapdragon processor. With 12GB of RAM, it ensures efficient multitasking. A 1-year Allstate warranty is included for added protection.', price=351.0, url='https://www.dealnews.com/products/Samsung/Unlocked-Samsung-Galaxy-S22-Ultra-512-GB-Phone/454156.html?iref=rss-c142')

In [18]:
from agents.scanner_agent import ScannerAgent

In [19]:
agent = ScannerAgent()
result = agent.scan()

In [20]:
result

DealSelection(deals=[Deal(product_description='The Refurbished Unlocked Apple iPhone 12 Pro Max features a spacious 256GB of storage, perfect for all your apps, photos, and videos. This smartphone boasts a stunning 6.7-inch Super Retina XDR display, enabling vibrant colors and sharp detail. With its powerful A14 Bionic chip, the device promises seamless performance across various tasks. The phone is also backed by a 1-year Allstate warranty for peace of mind.', price=400.0, url='https://www.dealnews.com/products/Apple/Unlocked-Apple-iPhone-12-Pro-Max-128-GB-Smartphone/314138.html?iref=rss-c142'), Deal(product_description="Experience cutting-edge technology with the Refurbished Unlocked Samsung Galaxy S22 Ultra, featuring a generous 512GB storage capacity. This smartphone is powered by a Qualcomm SM8450 octa-core Snapdragon processor and features a 6.8-inch 120Hz AMOLED display, providing vibrant colors and smooth visuals. Enhanced with 12GB of RAM and running on Android OS, it's design